# Fine-tune the SkillBridge matcher

## Introduction

The pretrained `all-MiniLM-L6-v2` matcher reached 89.7% (52/58) on the original hand-labeled set. It struggled with short jargon such as ETL and CI/CD, and confused System Design with Cloud Architecture. This notebook tests whether domain-specific phrasing helps. It does not change the app default.


## Load and inspect the dataset

The fixture contains the original 58 examples plus new examples covering all 67 canonical skills and close-but-distinct pairs. A fixed-seed 80/20 split is created here. Only the training indices are used for optimization; the held-out indices are used for both reported evaluations.


In [ ]:
import json, random
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
DATA_PATH = Path("tests/fixtures/matching_examples.json")
VOCAB_PATH = Path("knowledge_base/skills.json")
MODEL_OUTPUT = Path("models/skill_matcher_finetuned")
examples = json.loads(DATA_PATH.read_text(encoding="utf-8"))
vocabulary = [item["name"] for item in json.loads(VOCAB_PATH.read_text(encoding="utf-8"))["skills"]]
assert all(row["skill"] in vocabulary for row in examples)
indices = list(range(len(examples)))
random.Random(SEED).shuffle(indices)
eval_size = max(1, round(len(indices) * 0.2))
# Keep the specific known failure probes in the held-out set for direct comparison.
held_out_probes = {
    "Bash", "ETL", "CI/CD", "Data Warehousing", "Probability",
    "designed a scalable distributed system architecture",
    "designed secure and scalable cloud systems",
}
probe_indices = {i for i, row in enumerate(examples) if row["text"] in held_out_probes}
assert len(probe_indices) == len(held_out_probes)
# Keep the total held-out size at 20%, while guaranteeing the known probes are measured.
eval_indices = set(list(probe_indices) + [i for i in indices if i not in probe_indices][:eval_size-len(probe_indices)])
train_rows = [row for i, row in enumerate(examples) if i not in eval_indices]
eval_rows = [row for i, row in enumerate(examples) if i in eval_indices]
print(f"Total: {len(examples)} | train: {len(train_rows)} | held-out eval: {len(eval_rows)}")
pd.DataFrame(train_rows).head(8)


## Preprocessing

Each labeled phrase becomes a positive `(phrase, canonical skill)` pair with cosine target 1.0. It also receives one deterministic wrong-skill pair with target 0.0. This `CosineSimilarityLoss` setup fits one known positive label per phrase and explicitly teaches separation; the seeded negatives make results repeatable. Similar skills (including Java/JavaScript, SQL/NoSQL, and System Design/Cloud Architecture) appear as wrong-skill contrasts when selected. Only training rows are converted to training examples.


In [ ]:
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader

rng = random.Random(SEED)
hard_negative = {
    "SQL": "NoSQL", "NoSQL": "SQL", "Java": "JavaScript", "JavaScript": "Java",
    "System Design": "Cloud Architecture", "Cloud Architecture": "System Design",
}
train_examples = []
for row in train_rows:
    train_examples.append(InputExample(texts=[row["text"], row["skill"]], label=1.0))
    wrong_skill = hard_negative.get(row["skill"]) or rng.choice([skill for skill in vocabulary if skill != row["skill"]])
    train_examples.append(InputExample(texts=[row["text"], wrong_skill], label=0.0))
print(f"Training pairs (positive + negative): {len(train_examples)}")


## Baseline evaluation

Top-1 accuracy is the fraction of held-out phrases whose most similar canonical skill is the label. This reports ranking accuracy and does not apply the match rejection threshold. The new baseline can differ from the historical 89.7% because it uses this new held-out split.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

BASE_MODEL = "all-MiniLM-L6-v2"
baseline_model = SentenceTransformer(BASE_MODEL)

def evaluate_top1(model, rows):
    skill_vectors = model.encode(vocabulary, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    phrase_vectors = model.encode([row["text"] for row in rows], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    predicted = np.asarray(skill_vectors)[np.argmax(np.asarray(phrase_vectors) @ np.asarray(skill_vectors).T, axis=1)]
    expected = np.asarray([row["skill"] for row in rows])
    return {"accuracy": float(np.mean(predicted == expected)), "predicted": predicted, "expected": expected}

baseline = evaluate_top1(baseline_model, eval_rows)
print(f"Pretrained held-out top-1: {baseline['accuracy']:.1%} ({sum(baseline['predicted'] == baseline['expected'])}/{len(eval_rows)})")


## Fine-tuning

Fine-tune for four epochs with batch size 16 and a small learning rate. CPU is supported; runtime depends on hardware and the initial model download. The printed epoch losses come from the training split only.


In [ ]:
train_loader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(baseline_model)
baseline_model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=4,
    warmup_steps=0,
    optimizer_params={"lr": 2e-5},
    show_progress_bar=True,
    use_amp=False,
)
print("Training complete (see epoch loss output above).")


## Save the fine-tuned model

The model weights are saved locally under `models/skill_matcher_finetuned/`. This directory is Git-ignored because model weights are large; regenerate it by running this notebook after cloning.


In [ ]:
MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
baseline_model.save(str(MODEL_OUTPUT))
print(f"Saved model to {MODEL_OUTPUT}")


## Fine-tuned evaluation

Run the same held-out examples and canonical vocabulary used for baseline evaluation.


In [ ]:
finetuned_model = SentenceTransformer(str(MODEL_OUTPUT))
finetuned = evaluate_top1(finetuned_model, eval_rows)
print(f"Fine-tuned held-out top-1: {finetuned['accuracy']:.1%} ({sum(finetuned['predicted'] == finetuned['expected'])}/{len(eval_rows)})")


## Before/after comparison

The category checks below use the exact short jargon phrases and explicit System Design/Cloud Architecture confusion cases from the fixture, but only when those examples land in the held-out split. The category result is marked unavailable if the held-out split contains no relevant example; no training examples are used to claim improvement.


In [ ]:
def category_report(rows, before, after):
    rows = list(rows)
    hard_jargon = {"Bash", "ETL", "CI/CD", "Data Warehousing", "Probability"}
    masks = {
        "Short jargon": np.array([row["text"] in hard_jargon for row in rows]),
        "System Design / Cloud Architecture": np.array([row["skill"] in {"System Design", "Cloud Architecture"} for row in rows]),
    }
    report = []
    for name, mask in masks.items():
        if not mask.any():
            report.append({"category": name, "baseline": "not represented", "fine_tuned": "not represented", "change": "unavailable"})
            continue
        b = float(np.mean(before["predicted"][mask] == before["expected"][mask]))
        a = float(np.mean(after["predicted"][mask] == after["expected"][mask]))
        report.append({"category": name, "baseline": f"{b:.1%}", "fine_tuned": f"{a:.1%}", "change": "improved" if a>b else "worse" if a<b else "unchanged"})
    return report

comparison = pd.DataFrame([
    {"model": "Pretrained baseline", "held-out top-1 accuracy": f"{baseline['accuracy']:.1%}"},
    {"model": "Fine-tuned", "held-out top-1 accuracy": f"{finetuned['accuracy']:.1%}"},
])
comparison, pd.DataFrame(category_report(eval_rows, baseline, finetuned))


## Conclusion

Use the measured table above to record what this run showed. Fine-tuning may improve, leave unchanged, or reduce held-out accuracy; judge it by the held-out figures and category checks rather than training loss. The default `SkillMatcher()` still loads the pretrained hub model. To opt in after running this notebook, pass `model_name="models/skill_matcher_finetuned"`.
